# DAT404 — AI-Powered Data Quality Detective
## Main engineering notebook

**Model:** Ollama + Qwen3 (`qwen3:8b`)  
**Purpose:** Build, inspect, test and explain a tool-using intelligent agent.

This notebook is the main project workbook. The Streamlit application is in `app.py`.
Run the notebook from the project root so imports and relative paths work.

## 1. Problem definition

Tabular datasets can contain missing values, duplicate records, constant columns,
inconsistent text and unusual numeric values. These problems can reduce the reliability
of analysis. The proposed agent investigates a dataset and recommends safe next steps.

**Target users:** students, data analysts and small teams working with CSV/Excel files.

**Objectives**
1. Accept CSV and Excel data.
2. Use Qwen3 to plan a sequence of analysis tools.
3. Execute deterministic Python tools.
4. Explain findings using evidence from the tools.
5. retain episodic memory and user feedback.
6. Evaluate the agent using repeatable test cases.

## 2. PEAS description

| PEAS element | Project description |
|---|---|
| Performance | Correct tool execution, issue detection, grounded explanation, task success, low failure and latency |
| Environment | Local Python environment, uploaded files, Ollama service and user |
| Actuators | Tool execution, dashboard, report, recommendations and memory updates |
| Sensors | User goal, dataset values, schema, quality statistics and feedback |

The environment is partially observable because the agent cannot know the meaning of every column
or whether a statistical outlier is a real-world error without domain context.

## 3. Agent architecture and loop

```mermaid
flowchart TD
 U[User + CSV/XLSX] --> P[Perception]
 P --> Q[Qwen3 planner]
 Q --> V[Validate tool plan]
 V --> T[Python analysis tools]
 T --> O[Observe results + trace]
 O --> X[Qwen3 explanation]
 X --> F[User feedback]
 F --> M[(SQLite episodic memory)]
 M --> Q
```

The LLM proposes a plan, but it cannot execute arbitrary code. The Python application checks
every requested tool against an allow-list. Deterministic functions calculate data facts.

## 4. Environment setup

Install dependencies from `requirements.txt`. Install Ollama separately and pull Qwen3.
Keep the Ollama service running when executing LLM cells or the Streamlit application.

In [1]:
# Run once in a terminal from the project folder:
# pip install -r requirements.txt
# ollama pull qwen3:8b
# ollama run qwen3:8b

import pandas as pd
from pathlib import Path
from tools import profile_dataset, detect_quality_issues, detect_anomalies, create_cleaning_preview

DATA_PATH = Path("data/sample_dataset.csv")
df = pd.read_csv(DATA_PATH)
display(df.head())
print("Shape:", df.shape)

,Student_ID,Department,Attendance,Study_Hours,Exam_Score
0,S001,DSDA,92.0,4,85
1,S002,English,88.0,3,78
2,S003,DSDA,NaN,5,91
3,S004,Science,75.0,2,65
4,S005,DSDA,150.0,6,98


Shape: (7, 5)


## 5. Perception: inspect the dataset

The agent receives the uploaded dataframe and user goal. The following cell demonstrates
the data facts before planning. These outputs are generated by Python, not guessed by Qwen3.

In [2]:
profile = profile_dataset(df)
quality = detect_quality_issues(df)
print("PROFILE")
display(pd.Series({k:v for k,v in profile.items() if k not in ["numeric_summary"]}))
print("QUALITY ISSUES")
display(pd.Series(quality))

PROFILE


rows                                                               7
columns                                                            5
column_names       [Student_ID, Department, Attendance, Study_Hou...
data_types         {'Student_ID': 'str', 'Department': 'str', 'At...
memory_usage_kb                                                 0.47
dtype: object

QUALITY ISSUES


missing_values         {'Attendance': {'count': 1, 'percentage': 14.29}}
duplicate_rows                                                         0
constant_columns                                                      []
blank_text_values                                                     {}
total_missing_cells                                                    1
dtype: object

## 6. Tool library

The four tools have separate responsibilities:
- profiling provides structural information;
- quality checks find missingness, duplicates, constant columns and blank strings;
- IQR screening identifies potential numeric outliers;
- cleaning preview suggests actions but never changes the source dataframe.

In [3]:
anomalies = detect_anomalies(df)
cleaning_preview = create_cleaning_preview(df)
print("ANOMALY SCREENING")
display(pd.Series(anomalies))
print("CLEANING PREVIEW")
display(pd.Series(cleaning_preview))

ANOMALY SCREENING


method                                               1.5 x IQR rule
interpretation                    Candidates require domain review.
columns           {'Attendance': {'lower_bound': 70.375, 'upper_...
dtype: object

CLEANING PREVIEW


source_modified                                                  False
suggested_actions    [{'column': 'Attendance', 'issue': 'missing va...
message              This is a review plan only. The original datas...
dtype: object

## 7. Connect the Ollama Qwen3 agent

The next cell calls the same agent implementation used by `app.py`. If Ollama is not running,
the application records the issue and uses a deterministic fallback plan for tool execution.
The explanation call may be unavailable until Ollama is started.

In [6]:
from agent import DataQualityAgent

agent = DataQualityAgent(
    model="qwen3:8b",
    ollama_url="http://localhost:11434"
)

result = agent.investigate(
    df,
    "Investigate this student dataset, find data quality issues and potential outliers, and recommend safe next steps."
)

print("PLAN")
display(pd.Series(result["plan"]))

print("TRACE")
display(pd.DataFrame(result["trace"]))

print("QWEN3 EXPLANATION")
print(result["summary"])

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))

## 8. Memory and feedback

Episodes store the goal and investigation result in SQLite. Feedback stores a rating,
comment and issue category. This is a lightweight episodic memory mechanism. It is not
a model fine-tuning process: the model weights do not change.

In [ ]:
agent.record_feedback(
    rating=4,
    comment="The missing-value and outlier findings were useful.",
    issue_category="outliers"
)
print("Feedback guidance for later investigations:")
print(agent.feedback_guidance())
print("Recent episodes:")
display(pd.DataFrame(agent.memory.recent_episodes()))

## 9. Evaluation methodology

Use `evaluation/test_cases.csv`. Run every test, record actual results in
`evaluation/EVALUATION_LOG_TEMPLATE.md`, then calculate metrics.

- Task completion rate = completed test cases / total test cases × 100
- Tool execution success rate = successful tool calls / attempted tool calls × 100
- Tool selection accuracy = correct tool selections / expected selections × 100
- Mean latency = total elapsed seconds / completed investigations

Do not insert expected results as if they were measured results.

In [ ]:
# Example calculation after entering your REAL measured counts:
def percentage(numerator, denominator):
    return round((numerator / denominator) * 100, 2) if denominator else 0.0

# Replace these placeholders after conducting your experiments.
completed_cases = 0
total_cases = 10
successful_tool_calls = 0
attempted_tool_calls = 0

print("Task completion rate (%):", percentage(completed_cases, total_cases))
print("Tool execution success rate (%):", percentage(successful_tool_calls, attempted_tool_calls))

## 10. Visualisation example

This chart visualises the IQR candidate counts produced by the Python tool.
Interpret it as a screening result, not proof of incorrect records.

In [ ]:
import matplotlib.pyplot as plt

anomaly_columns = anomalies.get("columns", {})
if anomaly_columns:
    names = list(anomaly_columns.keys())
    counts = [anomaly_columns[name]["candidate_count"] for name in names]
    plt.figure(figsize=(8, 4))
    plt.bar(names, counts)
    plt.title("Potential outlier candidates by numeric column")
    plt.xlabel("Numeric column")
    plt.ylabel("Candidate count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No eligible numeric columns or no IQR candidates.")

## 11. Safety, limitations and reflection

- The original dataframe is not changed by the cleaning-preview tool.
- IQR thresholds may flag valid rare observations.
- Qwen3 may hallucinate; explanations must be checked against tool outputs.
- The allow-list reduces the risk of arbitrary tool execution.
- Local inference supports a local workflow, but file access and privacy still depend on the machine and user practices.
- Feedback changes stored priorities/guidance, not Qwen3 model weights.
- The current system has no RAG or multi-agent collaboration; these are not necessary for this focused use case.

## 12. Final reflection

After testing, explain which test cases succeeded, which failed, what caused the failures,
what you changed, and what remains limited. Include your own screenshots, measured metrics,
references and evidence of progressive development.